<a href="https://colab.research.google.com/github/prvnabna/Ayurveda-Knowledge-Graph/blob/main/ayurveda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

file1 = "/content/Ashwagandha Annotated_Entities.xlsx"
file2 = "/content/Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx"
file3 = "/content/Clinical_Evaluation Annotated_Entities.xlsx"

for file in [file1, file2, file3]:
    print("\n====================")
    print(file)

    df = pd.read_excel(file)

    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")
    print(df.head())


/content/Ashwagandha Annotated_Entities.xlsx
Columns:
['Sentence', 'Entity', 'Label']

First 5 rows:
                                    Sentence              Entity    Label
0       Ashwagandha promotes health benefits         Ashwagandha     HERB
1       Ashwagandha promotes health benefits  Withania somnifera     HERB
2       Ashwagandha promotes health benefits       winter cherry     HERB
3           Ashwagandha has effects on sleep               sleep  SYMPTOM
4  Ashwagandha has anti-inflammatory effects        inflammation  SYMPTOM

/content/Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx
Columns:
['Sentence', 'Entity', 'Label']

First 5 rows:
                                            Sentence  \
0  Diabetes mellitus has emerged as a global heal...   
1  People with impaired glucose tolerance may dev...   
2       Diabetes complications include heart disease   
3       Diabetes complications include renal failure   
4  Diabetes complications include neurological di...

In [ ]:
import pandas as pd

file1 = "/content/Ashwagandha Annotated_Entities.xlsx"
file2 = "/content/Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx"
file3 = "/content/Clinical_Evaluation Annotated_Entities.xlsx"

df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)
df3 = pd.read_excel(file3)

df = pd.concat([df1, df2, df3], ignore_index=True)

print("Total rows:", len(df))
df.head()

Total rows: 576


,Sentence,Entity,Label
0,Ashwagandha promotes health benefits,Ashwagandha,HERB
1,Ashwagandha promotes health benefits,Withania somnifera,HERB
2,Ashwagandha promotes health benefits,winter cherry,HERB
3,Ashwagandha has effects on sleep,sleep,SYMPTOM
4,Ashwagandha has anti-inflammatory effects,inflammation,SYMPTOM


In [ ]:
TRAIN_DATA = []

for sentence in df["Sentence"].unique():

    rows = df[df["Sentence"] == sentence]

    entities = []

    for _, row in rows.iterrows():

        entity = str(row["Entity"]).strip()
        label = str(row["Label"]).strip()

        start = sentence.find(entity)

        if start != -1:

            end = start + len(entity)

            entities.append((start, end, label))

    TRAIN_DATA.append(
        (
            sentence,
            {"entities": entities}
        )
    )

print("Total sentences:", len(TRAIN_DATA))
print(TRAIN_DATA[0])

Total sentences: 541
('Ashwagandha promotes health benefits', {'entities': [(0, 11, 'HERB')]})


In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans

nlp = spacy.blank("en")

db = DocBin()

for text, annotations in TRAIN_DATA:

    doc = nlp.make_doc(text)

    spans = []

    for start, end, label in annotations["entities"]:

        span = doc.char_span(
            start,
            end,
            label=label,
            alignment_mode="expand"
        )

        if span is not None:
            spans.append(span)

    spans = filter_spans(spans)

    doc.ents = spans

    db.add(doc)

print("Dataset created successfully")

Dataset created successfully


In [ ]:
db.to_disk("train.spacy")

In [ ]:
import os

print(os.path.exists("train.spacy"))

True


In [ ]:
!pip install -U spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!python -m spacy init config config.cfg \
--lang en \
--pipeline ner \
--force

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
!python -m spacy train config.cfg \
--output output_model \
--paths.train train.spacy \
--paths.dev train.spacy

✔ Created output directory: output_model
ℹ Saving to output directory: output_model
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     65.79    0.00    0.00    0.00    0.00
  7     200        198.12   4258.49   91.02   90.07   91.98    0.91
 17     400        204.05    750.18   98.70   98.59   98.82    0.99
 28     600        125.06    281.98   99.53   99.76   99.29    1.00
 43     800        175.30    174.36   99.76   99.76   99.76    1.00
 60    1000        186.10    116.42   99.88  100.00   99.76    1.00
 81    1200        228.68    103.91   99.88  100.00   99.76    1.00
106    1400        109.68     82.60   99.88  100.00 

In [ ]:
import spacy

nlp = spacy.load("output_model/model-best")

In [ ]:
text = """
Ashwagandha helps reduce oxidative stress and improve sleep.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, ent.label_)

reduce oxidative stress CHEMICAL
improve sleep. SYMPTOM


In [ ]:
!python -m spacy init config config.cfg --lang en --pipeline ner --force

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
!ls

'Ashwagandha Annotated_Entities.xlsx'			    output_model
'Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx'   sample_data
'Clinical_Evaluation Annotated_Entities.xlsx'		    train.spacy
 config.cfg


In [ ]:
!python -m spacy train config.cfg \
--output output_model \
--paths.train train.spacy \
--paths.dev train.spacy

ℹ Saving to output directory: output_model
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     65.79    0.00    0.00    0.00    0.00
  7     200        198.12   4258.49   91.02   90.07   91.98    0.91
 17     400        204.05    750.18   98.70   98.59   98.82    0.99
 28     600        125.06    281.98   99.53   99.76   99.29    1.00
 43     800        175.30    174.36   99.76   99.76   99.76    1.00
 60    1000        186.10    116.42   99.88  100.00   99.76    1.00
 81    1200        228.68    103.91   99.88  100.00   99.76    1.00
106    1400        109.68     82.60   99.88  100.00   99.76    1.00
137    1600        334.65

In [ ]:
!ls output_model

model-best  model-last


In [ ]:
import spacy

nlp = spacy.load("output_model/model-best")

text = """
Ashwagandha improves sleep and reduces oxidative stress.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "---->", ent.label_)

sleep ----> SYMPTOM
oxidative stress. ----> DISEASE


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive"

'Colab Notebooks'  'Untitled document.gdoc'


In [ ]:
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/Annotated Data"
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/Models"
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/output"
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/Reports"
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/semantic Tags"

In [ ]:
!ls "/content/drive/MyDrive"

 Ayurveda_NLP_Project  'Colab Notebooks'  'Untitled document.gdoc'


In [ ]:
!ls "/content/drive/MyDrive/Ayurveda_NLP_Project"

'Annotated Data'   Models   output   Reports  'semantic Tags'


In [ ]:
!ls

'Ashwagandha Annotated_Entities.xlsx'			    drive
'Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx'   output_model
'Clinical_Evaluation Annotated_Entities.xlsx'		    sample_data
 config.cfg						    train.spacy


In [ ]:
!cp train.spacy "/content/drive/MyDrive/Ayurveda_NLP_Project/Models/"

In [ ]:
!ls "/content/drive/MyDrive/Ayurveda_NLP_Project/Models"

train.spacy


In [ ]:
!ls

'Ashwagandha Annotated_Entities.xlsx'			    drive
'Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx'   output_model
'Clinical_Evaluation Annotated_Entities.xlsx'		    sample_data
 config.cfg						    train.spacy


In [ ]:
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/Models"
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/output"
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/Reports"
!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/semantic Tags"

In [ ]:
!ls "/content/drive/MyDrive/Ayurveda_NLP_Project"

'Annotated Data'   Models   output   Reports  'semantic Tags'


In [ ]:
!cp -r output_model "/content/drive/MyDrive/Ayurveda_NLP_Project/Models/"

In [ ]:
!cp train.spacy "/content/drive/MyDrive/Ayurveda_NLP_Project/Models/"

In [ ]:
!ls "/content/drive/MyDrive/Ayurveda_NLP_Project/Models"

output_model  train.spacy


In [ ]:
import spacy

nlp = spacy.load("output_model/model-best")

text = """
Ashwagandha improves sleep and reduces inflammation.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "---->", ent.label_)

sleep ----> SYMPTOM
reduces inflammation. ----> CHEMICAL


In [ ]:
import pandas as pd

results = []

for ent in doc.ents:
    results.append({
        "Entity": ent.text,
        "Label": ent.label_
    })

pd.DataFrame(results).to_csv(
"/content/drive/MyDrive/Ayurveda_NLP_Project/output/ner_results.csv",
index=False
)

In [ ]:
!ls "/content/drive/MyDrive/Ayurveda_NLP_Project/output"

ner_results.csv


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving _Ashwagandha Semantic_Tags (1).xlsx to _Ashwagandha Semantic_Tags (1) (1).xlsx


In [ ]:
!ls

'Ashwagandha Annotated_Entities.xlsx'			    config.cfg
'_Ashwagandha Semantic_Tags (1) (1).xlsx'		    drive
'_Ashwagandha Semantic_Tags (1).xlsx'			    output_model
'Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx'   sample_data
'Clinical_Evaluation Annotated_Entities.xlsx'		    train.spacy


In [ ]:
!cp "_Ashwagandha Semantic_Tags.xlsx" "/content/drive/MyDrive/Ayurveda_NLP_Project/semantic Tags/"

cp: cannot stat '_Ashwagandha Semantic_Tags.xlsx': No such file or directory


In [ ]:
!ls

'Ashwagandha Annotated_Entities.xlsx'			    config.cfg
'_Ashwagandha Semantic_Tags (1) (1).xlsx'		    drive
'_Ashwagandha Semantic_Tags (1).xlsx'			    output_model
'Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx'   sample_data
'Clinical_Evaluation Annotated_Entities.xlsx'		    train.spacy


In [ ]:
!cp "/content/_Ashwagandha Semantic_Tags (1).xlsx" "/content/_Ashwagandha Semantic_Tags (1) (1).xlsx"

In [ ]:
import os

for f in os.listdir():
    print(f)

.config
Ashwagandha Annotated_Entities.xlsx
Ayurveda_Management_of_Diabetes Annotated_Entities.xlsx
output_model
config.cfg
Clinical_Evaluation Annotated_Entities.xlsx
train.spacy
drive
_Ashwagandha Semantic_Tags (1).xlsx
_Ashwagandha Semantic_Tags (1) (1).xlsx
sample_data


In [ ]:
import pandas as pd

semantic = pd.read_excel(
"/content/_Ashwagandha Semantic_Tags (1).xlsx"
)

print("Columns:")
print(semantic.columns.tolist())

semantic.head()

Columns:
['Term', 'Category', 'Tag1', 'Tag2', 'Tag3', 'Tag4', 'Tag5']


,Term,Category,Tag1,Tag2,Tag3,Tag4,Tag5
0,Ashwagandha,HERB,ADAPTOGEN,STRESS_RELIEF,SLEEP_AID,IMMUNITY,ANTI_INFLAMMATORY
1,Withania somnifera,HERB,ALIAS,ASHWAGANDHA,NaN,NaN,NaN
2,winter cherry,HERB,ALIAS,ASHWAGANDHA,NaN,NaN,NaN
3,sleep,SYMPTOM,SLEEP_RELATED,REST_FUNCTION,NaN,NaN,NaN
4,inflammation,SYMPTOM,IMMUNE_RESPONSE,PAIN_RELATED,NaN,NaN,NaN


In [ ]:
from google.colab import files

files.download("train.spacy")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r output_model.zip output_model

  adding: output_model/ (stored 0%)
  adding: output_model/model-best/ (stored 0%)
  adding: output_model/model-best/tokenizer (deflated 81%)
  adding: output_model/model-best/config.cfg (deflated 61%)
  adding: output_model/model-best/ner/ (stored 0%)
  adding: output_model/model-best/ner/model (deflated 7%)
  adding: output_model/model-best/ner/moves (deflated 69%)
  adding: output_model/model-best/ner/cfg (deflated 33%)
  adding: output_model/model-best/vocab/ (stored 0%)
  adding: output_model/model-best/vocab/lookups.bin (stored 0%)
  adding: output_model/model-best/vocab/vectors (deflated 45%)
  adding: output_model/model-best/vocab/strings.json (deflated 71%)
  adding: output_model/model-best/vocab/key2row (stored 0%)
  adding: output_model/model-best/vocab/vectors.cfg (stored 0%)
  adding: output_model/model-best/meta.json (deflated 69%)
  adding: output_model/model-best/tok2vec/ (stored 0%)
  adding: output_model/model-best/tok2vec/model (deflated 8%)
  adding: output_model/mo

In [ ]:
from google.colab import files

files.download("output_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r output_model.zip output_model

updating: output_model/ (stored 0%)
updating: output_model/model-best/ (stored 0%)
updating: output_model/model-best/tokenizer (deflated 81%)
updating: output_model/model-best/config.cfg (deflated 61%)
updating: output_model/model-best/ner/ (stored 0%)
updating: output_model/model-best/ner/model (deflated 7%)
updating: output_model/model-best/ner/moves (deflated 69%)
updating: output_model/model-best/ner/cfg (deflated 33%)
updating: output_model/model-best/vocab/ (stored 0%)
updating: output_model/model-best/vocab/lookups.bin (stored 0%)
updating: output_model/model-best/vocab/vectors (deflated 45%)
updating: output_model/model-best/vocab/strings.json (deflated 71%)
updating: output_model/model-best/vocab/key2row (stored 0%)
updating: output_model/model-best/vocab/vectors.cfg (stored 0%)
updating: output_model/model-best/meta.json (deflated 69%)
updating: output_model/model-best/tok2vec/ (stored 0%)
updating: output_model/model-best/tok2vec/model (deflated 8%)
updating: output_model/mo

In [ ]:
from google.colab import files
files.download('output_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('train.spacy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('/content/_Ashwagandha Semantic_Tags (1).xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/Ayurveda_NLP_Project/Models"

!cp train.spacy "/content/drive/MyDrive/Ayurveda_NLP_Project/Models/"
!cp -r output_model "/content/drive/MyDrive/Ayurveda_NLP_Project/Models/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!find "/content/drive/MyDrive/Ayurveda_NLP_Project" -type f

/content/drive/MyDrive/Ayurveda_NLP_Project/Models/train.spacy
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/tokenizer
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/meta.json
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/config.cfg
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/tok2vec/cfg
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/tok2vec/model
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/ner/model
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/ner/moves
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/ner/cfg
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/vocab/strings.json
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model-last/vocab/vectors
/content/drive/MyDrive/Ayurveda_NLP_Project/Models/output_model/model